# 🛩️ UAV Training — YOLO Detection

Bu notebook, UAV tespit modelini Google Colab üzerinde eğitmek için tek hücrede çalışan bootstrap script'ini içerir.

**Adımlar:**
1. Google Drive'ı bağla
2. GitHub repo'yu klonla
3. Bağımlılıkları kur
4. Dataset'i Drive → yerel SSD'ye aktar (tar + pv)
5. Eğitimi başlat (checkpoint varsa otomatik devam)

> ⚠️ Aşağıdaki hücrede `REPO_URL` değişkenini kendi GitHub repo adresinizle değiştirin.

In [ ]:
##############################################################################
# \ud83d\ude80 YOLO UAV Training Bootstrap \u2014 Google Colab
# Paste this entire cell into Colab and run.
##############################################################################

# \u2500\u2500 Configuration \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
REPO_URL      = \"https://github.com/YOUR_USER/YOUR_REPO.git\"   # \u2190 set this\n",
REPO_BRANCH   = \"main\"                                          # \u2190 set this\n",
DRIVE_DATASET = \"/content/drive/MyDrive/train\"\n",
LOCAL_CACHE   = \"/content/train_local\"\n",
DRIVE_RUNS    = \"/content/drive/MyDrive/runs\"\n",
TRAIN_SCRIPT  = \"uav_training/train.py\"\n",
# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n",
\n",
import subprocess, sys, os, pathlib, glob, textwrap\n",
\n",
def _run(cmd: str, *, check: bool = True, shell: bool = True, **kw):\n",
    r = subprocess.run(cmd, shell=shell, check=check, **kw)\n",
    return r\n",
\n",
def _banner(msg: str):\n",
    print(f\"\\n{'='*60}\\n  {msg}\\n{'='*60}\")\n",
\n",
# 1. Mount Google Drive\n",
_banner(\"1/6 \u2014 Mounting Google Drive\")\n",
from google.colab import drive\n",
drive.mount(\"/content/drive\", force_remount=False)\n",
\n",
if not os.path.isdir(DRIVE_DATASET):\n",
    raise FileNotFoundError(f\"Dataset not found at {DRIVE_DATASET}.\")\n",
os.makedirs(DRIVE_RUNS, exist_ok=True)\n",
print(f\"  \u2713 Drive mounted\")\n",
\n",
# 2. Clone or Pull Repository\n",
_banner(\"2/6 \u2014 Setting up repository\")\n",
REPO_DIR = \"/content/repo\"\n",
if os.path.isdir(os.path.join(REPO_DIR, \".git\")):\n",
    _run(f\"git -C {REPO_DIR} fetch --all --quiet\")\n",
    _run(f\"git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}\")\n",
else:\n",
    _run(f\"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}\")\n",
print(\"  \u2713 Repo ready\")\n",
\n",
# 3. Install Requirements\n",
_banner(\"3/6 \u2014 Installing dependencies\")\n",
req_file = os.path.join(REPO_DIR, \"requirements.txt\")\n",
if os.path.isfile(req_file):\n",
    _run(f\"{sys.executable} -m pip install -q -r {req_file}\")\n",
_run(f\"{sys.executable} -m pip install -q ultralytics\", check=False)\n",
print(\"  \u2713 Dependencies installed\")\n",
\n",
# 4. Dataset Streaming\n",
_banner(\"4/6 \u2014 Preparing dataset cache\")\n",
_run(\"which pv >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq pv > /dev/null 2>&1)\")\n",
if os.path.isdir(LOCAL_CACHE) and os.listdir(LOCAL_CACHE):\n",
    print(f\"  \u2713 Cache exists \u2014 skipping\")\n",
else:\n",
    os.makedirs(LOCAL_CACHE, exist_ok=True)\n",
    du = subprocess.run(f'du -sb \"{DRIVE_DATASET}\" | cut -f1', shell=True, capture_output=True, text=True)\n",
    total = du.stdout.strip()\n",
    pv_flag = f\"-s {total}\" if total.isdigit() else \"\"\n",
    _run(f'tar -C \"{DRIVE_DATASET}\" -cf - . | pv -f -pterb {pv_flag} | tar -C \"{LOCAL_CACHE}\" -xf -')\n",
    print(\"  \u2713 Streaming complete\")\n",
_run(f'df -h /content | tail -1')\n",
\n",
# 5. Configure Output\n",
_banner(\"5/6 \u2014 Configuring output\")\n",
_run(f'yolo settings runs_dir=\"{DRIVE_RUNS}\"', check=False)\n",
\n",
# 6. Launch Training\n",
_banner(\"6/6 \u2014 Starting training\")\n",
def find_ckpt(d):\n",
    c = glob.glob(os.path.join(d, '**', 'weights', 'last.pt'), recursive=True)\n",
    return max(c, key=os.path.getmtime) if c else None\n",
ckpt = find_ckpt(DRIVE_RUNS)\n",
script = os.path.join(REPO_DIR, TRAIN_SCRIPT)\n",
if ckpt:\n",
    print(f\"  \ud83d\udd04 Resuming: {ckpt}\")\n",
    cmd = f'{sys.executable} \"{script}\" --resume \"{ckpt}\"'\n",
else:\n",
    print(\"  \ud83c\udd95 Fresh training\")\n",
    cmd = f'{sys.executable} \"{script}\" --data \"{LOCAL_CACHE}\"'\n",
os.chdir(REPO_DIR)\n",
_run(cmd)\n",
_banner(\"\u2705 Training complete\")\n"